# Verdius v1 Pipeline — Article-Level Causal Extraction

End-to-end inference pipeline running entirely in Colab.

**Pipeline stages:**
1. Sentence split — spaCy sentencizer with char offsets
2. st0 filter — binary causal classifier
3. st1 + st2 — relation type classifier + REBEL event-pair extraction
4. Coref — fastcoref / LingMess over the full article
5. Build graph JSON — assemble (Article, Events, Causal Edges) per article
6. Write to Neo4j — `:Article -[:CONTAINS]-> :Event -[:CAUSES|ENABLES|PREVENTS|INTENDS]-> :Event`

**Events:** ST2 uses `Babelscape/rebel-large` by default and graph event IDs are based on normalized event phrase text, so repeated event phrases can link across articles.


## 0. Confirm GPU

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU detected — switch runtime to GPU.'
print('Device:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. Install dependencies

`torch` ships with Colab. We add spacy (sentence split), transformers (st0/st1/st2), fastcoref (coref), neo4j (graph sink).

In [ ]:
!pip install -q spacy==3.8.14 transformers==4.44.2 fastcoref neo4j accelerate


## 2. Clone the repo

Public — no auth needed. Pulls latest if already cloned.

In [ ]:
import os

if not os.path.isdir('Relation_extraction'):
    !git clone https://github.com/eitang5/Relation_extraction.git
else:
    print('Repo already cloned. Pulling latest...')
    !git -C Relation_extraction pull

%cd Relation_extraction

# Use the HF REBEL backend for ST2.
os.environ['ST2_BACKEND'] = 'rebel'
os.environ.setdefault('ST2_REBEL_CKPT', 'Babelscape/rebel-large')
os.environ.setdefault('ST2_REBEL_MAX_LEN', '512')
os.environ.setdefault('ST2_REBEL_MAX_NEW_TOKENS', '128')


## 3. Mount Google Drive (for output)

Models load from HF Hub (no Drive needed for them). We still mount Drive so the final article JSON outputs persist across runtime restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

out_root = '/content/drive/MyDrive/verdius/v1_outputs'
os.makedirs(out_root, exist_ok=True)
print(f'Output dir ready: {out_root}')

## 4. Sample article

Short news-style article (Iran / Hormuz / oil) with several causal claims. Running example through every cell below.

In [ ]:
SAMPLE_ARTICLE = (
    "The Strait of Hormuz remained closed for a third consecutive day on Wednesday "
    "after Iranian naval forces detained a Liberian-flagged tanker on Monday morning, "
    "sending Brent crude to a 14-month high of $112 a barrel. Saudi Aramco said it "
    "would temporarily reroute roughly 2 million barrels per day through the "
    "East-West Pipeline because of the closure. The detention followed a strike by "
    "Israeli aircraft on a missile depot in western Iran on Sunday night, which "
    "Tehran blamed for triggering its retaliation. European equity markets fell "
    "sharply, with the DAX closing 2.4% lower as investors priced in higher input costs."
)

ARTICLE_ID = 'demo_iran_hormuz'
ARTICLE_METADATA = {
    'title': 'Strait of Hormuz closure sends oil prices higher',
    'url': 'https://example.com/demo',
    'publication_date': '2026-05-06',
}
print(f'Article length: {len(SAMPLE_ARTICLE)} chars')

## 5. Module 1 — sentence split

spaCy blank('en') + sentencizer. Char offsets are load-bearing for downstream span resolution.

In [ ]:
from v1_pipeline.sentence_split import split_sentences

sentences = split_sentences(SAMPLE_ARTICLE)
for i, s in enumerate(sentences):
    s['idx'] = i  # attach idx so we can trace through later stages
    print(f'  [{i}] [{s["start"]:>3}:{s["end"]:>3}] {s["text"][:80]!r}{"..." if len(s["text"]) > 80 else ""}')
print(f'\nTotal: {len(sentences)} sentences')

## 6. Module 2 — st0 filter

Binary causal classifier (RoBERTa-large). Drops sentences predicted as `no_relation`. Expected ~90% drop rate on real articles; demo article will keep most because it's causal-dense.

In [ ]:
from v1_pipeline.st0_filter import filter_sentences

survivors = filter_sentences(sentences)
print(f'{len(survivors)} / {len(sentences)} sentences survived st0')
for s in survivors:
    print(f'  [{s["idx"]}] {s["text"][:90]!r}{"..." if len(s["text"]) > 90 else ""}')

## 7. Module 3 — st1 relation labels + st2 REBEL event pairs

**st1** predicts the CausalSense relation type for each kept sentence.

**st2** uses `Babelscape/rebel-large` to extract one or more subject/object event pairs from the same sentences. The graph builder uses the ST2 event phrases as Neo4j event nodes and ST1's relation label as the typed edge.


In [ ]:
from v1_pipeline.st1_classify import classify_relations
from v1_pipeline.st2_extract import extract_triples_batch

rel_results = classify_relations(survivors)
kept_sentences = [r['sentence'] for r in rel_results]
triple_results = extract_triples_batch(kept_sentences)

print(f'After st1: {len(rel_results)} sentences kept')
for r, t in zip(rel_results, triple_results):
    triples = t.get('triples', [])
    print(f'  [{r["sentence"]["idx"]}] {r["relation_type"]:8} (p={r["confidence"]:.2f})  {len(triples)} ST2 triple(s)')
    for triple in triples:
        raw_rel = triple.get('raw_relation') or 'n/a'
        print(f'      {triple["subject"]!r} -> {triple["object"]!r}  [rebel={raw_rel!r}]')


## 8. Module 4 — coref

fastcoref / LingMess clusters NP-shaped mentions across the whole article. We use clusters only to MERGE event nodes (two spans in the same cluster → one Event node). No Entity layer in v1.

First call downloads a longformer-based coref model (~500MB), cached for the session.

In [ ]:
from v1_pipeline.coref import coref_article

clusters = coref_article(SAMPLE_ARTICLE)
print(f'{len(clusters)} coref clusters\n')
for ci, cluster in enumerate(clusters):
    mentions = [SAMPLE_ARTICLE[s:e] for s, e in cluster]
    print(f'  cluster {ci}: {mentions}')

## 9. Event keys

Event nodes are keyed by normalized ST2 event phrase text. That keeps readable event names in Neo4j and allows repeated event phrases to link across articles. Coref still runs over the full article and is reported for inspection.


In [ ]:
import hashlib

for r, t in zip(rel_results, triple_results):
    for triple in t.get('triples', []):
        for role in ('subject', 'object'):
            phrase = triple[role]
            key = ' '.join(phrase.casefold().split())
            event_id = 'event:' + hashlib.sha1(key.encode('utf-8')).hexdigest()[:16]
            print(f'  [{r["sentence"]["idx"]}] {role:7} {event_id}  {phrase!r}')


## 10. Module 7a — build article graph

Orchestrator: runs modules 1-6 end-to-end and emits the canonical article-graph dict (sentences + events + edges + stats). This is what gets written to disk as JSON and loaded into Neo4j.

In [ ]:
import json
from v1_pipeline.graph_build import build_article_graph

graph = build_article_graph(ARTICLE_ID, SAMPLE_ARTICLE, ARTICLE_METADATA)

print('Stats:', json.dumps(graph['stats'], indent=2))
print(f'\n{len(graph["events"])} events:')
for ev in graph['events']:
    print(f'  {ev["id"][-12:]}  {ev["name"]!r}  (in sentences {ev["sentence_indices"]})')
print(f'\n{len(graph["edges"])} causal edges:')
for e in graph['edges']:
    print(f'  {e["source"][-12:]} -[{e["relation_type"]}]-> {e["target"][-12:]}  (p={e["confidence"]:.2f})')

## 11. Save JSON and write to Neo4j

Two-step sink: persist the article graph as JSON to Drive (canonical audit log), then write the same data to Neo4j via `MERGE` (idempotent — safe to re-run).

**Set your dev Neo4j credentials below before running this cell.**

In [ ]:
import json
from pathlib import Path

# 1. Save JSON to Drive.
out_dir = Path('/content/drive/MyDrive/verdius/v1_outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'{ARTICLE_ID}.json'
out_path.write_text(json.dumps(graph, indent=2))
print(f'JSON written to {out_path} ({out_path.stat().st_size} bytes)')

# 2. Set Neo4j env vars (dev instance) here or from Colab secrets.
import os
try:
    from google.colab import userdata
    os.environ['NEO4J_URI'] = userdata.get('NEO4J_URI') or os.environ.get('NEO4J_URI', '')
    os.environ['NEO4J_PASSWORD'] = userdata.get('NEO4J_PASSWORD') or os.environ.get('NEO4J_PASSWORD', '')
except Exception:
    pass

os.environ.setdefault('NEO4J_USERNAME', 'neo4j')
os.environ.setdefault('NEO4J_DATABASE', 'eitan-v0')

if os.environ.get('NEO4J_URI') and os.environ.get('NEO4J_PASSWORD'):
    from v1_pipeline.neo4j_writer import write_article_to_neo4j
    counts = write_article_to_neo4j(graph)
    print(f'Neo4j writes: {counts}')
else:
    print('Skipping Neo4j write — NEO4J_URI / NEO4J_PASSWORD not set.')


## 12. Module 8 — multi-article runner

CLI form for batch jobs. Reads a JSONL where each line is `{"id": "...", "text": "...", ...metadata}`, writes one JSON per article to `--output-dir`, optionally to Neo4j.

Demo: write the sample article as a 1-line JSONL and process it.

In [ ]:
import json
from pathlib import Path

demo_jsonl = Path('/content/demo_articles.jsonl')
demo_jsonl.write_text(json.dumps({
    'id': ARTICLE_ID,
    'text': SAMPLE_ARTICLE,
    **ARTICLE_METADATA,
}) + '\n')

# Without --neo4j (JSON only). Add --neo4j once Neo4j env vars are set.
!python -m v1_pipeline.runner --input /content/demo_articles.jsonl --output-dir /content/drive/MyDrive/verdius/v1_outputs

## Next steps

**v1 done if every cell above ran cleanly.**

Output you should see:
- Sample article splits into ~5-7 sentences
- st0 keeps most of them (article is causal-dense)
- st1 emits `cause`/`enable`/`prevent`/`intend` with confidences
- st2 emits REBEL event pairs as readable phrases
- Coref clusters print
- Final graph has `Article`, `Event`, `CONTAINS`, and typed causal edges
- JSON written to Drive
- Neo4j write counts if credentials are configured
